In [ ]:
# ── Imports & paths ────────────────────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
sys.path.insert(0, '.')
from utils import DATA_DIR, OUT_DIR

EASTERN_PROVINCES = ['Nord-kivu', 'Sud-kivu', 'Ituri']

CAUSE_TRANSLATIONS = {
    'Attaques, affrontements armées':        'Armed attacks / clashes',
    'Conflits fonciers, intercommunautaires':'Land / intercommunal conflict',
    'Catastrophe naturelle':                 'Natural disaster',
    'Amélioration des conditions':           'Improved conditions',
    'AméLioration des conditions':           'Improved conditions',
    'AmÃ©lioration des conditions':          'Improved conditions',
    'Epidemie':                              'Epidemic',
    'Autre':                                 'Other',
    'Malnutrition':                          'Malnutrition',
}

# Shared style
BG     = 'white'
FG     = 'black'
ACCENT = '#444444'
RULE   = '#aaaaaa'


# ── Helper functions ───────────────────────────────────────────────────────

def load_and_clean(path, eastern_provinces, cause_map):
    """Load an OCHA movement CSV, filter to eastern provinces and matching snapshot months,
    deduplicate by event ID, and translate cause labels to English."""
    df = pd.read_csv(path)
    df['movement_date']  = pd.to_datetime(df['movement_date'],  errors='coerce')
    df['snapshot_month'] = pd.to_datetime(df['snapshot_month'])

    # Keep only events whose movement date falls within the snapshot month
    df = df[
        (df['movement_date'].dt.year  == df['snapshot_month'].dt.year) &
        (df['movement_date'].dt.month == df['snapshot_month'].dt.month)
    ]
    # One row per event
    df = df.sort_values('snapshot_month').drop_duplicates(subset=['id'], keep='first')
    # Eastern DRC only
    df = df[df['admin1_label'].isin(eastern_provinces)]
    # Translate causes
    df['cause_label'] = df['cause_label'].str.strip().replace(cause_map)
    return df


def monthly_totals(df, col='person'):
    """Sum person counts by snapshot_month."""
    return df.groupby('snapshot_month')[col].sum()


def save_fig(fig, path):
    """Save figure to path and close."""
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f'Saved → {path}')


def make_summary_png(dep, ret, out_path):
    """Export key summary statistics as a formatted PNG table."""
    total_dep = dep['person'].sum()
    total_ret = ret['person'].sum()
    net       = total_dep - total_ret
    date_min  = dep['snapshot_month'].min().strftime('%b %Y')
    date_max  = dep['snapshot_month'].max().strftime('%b %Y')
    top_cause_dep = dep['cause_label'].value_counts().index[0]
    top_cause_ret = ret['cause_label'].value_counts().index[0]

    rows = [
        ('Total displaced (persons)',  f'{total_dep:,.0f}'),
        ('Total returnees (persons)',  f'{total_ret:,.0f}'),
        ('Net displacement',           f'{net:+,.0f}'),
        ('Date range',                 f'{date_min} → {date_max}'),
        ('Top displacement cause',     top_cause_dep),
        ('Top return cause',           top_cause_ret),
    ]

    fig, ax = plt.subplots(figsize=(9, 4.2))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.text(0.5, 0.97, 'Displacement Summary — Eastern DRC (2021–2026)',
            transform=ax.transAxes, color=FG, fontsize=12,
            fontweight='bold', ha='center', va='top')

    Y0, ROW_H = 0.80, 0.115

    def hline(y, lw=0.8, color=RULE):
        ax.plot([0.03, 0.97], [y, y], color=color, linewidth=lw,
                transform=ax.transAxes, clip_on=False)

    hline(Y0, lw=1.2, color='black')
    y = Y0 - 0.01
    for label, value in rows:
        ax.text(0.04, y, label, transform=ax.transAxes, color=ACCENT,
                fontsize=9.5, va='top', ha='left', fontfamily='monospace')
        ax.text(0.97, y, value, transform=ax.transAxes, color=FG,
                fontsize=9.5, va='top', ha='right', fontfamily='monospace', fontweight='bold')
        y -= ROW_H

    hline(y + 0.02, lw=1.2, color='black')
    ax.text(0.04, y - 0.01,
            'Source: OCHA IDP monitoring data. Eastern provinces: Nord-Kivu, Sud-Kivu, Ituri.',
            transform=ax.transAxes, color=ACCENT, fontsize=7.5, va='top', fontfamily='monospace')

    save_fig(fig, out_path)


In [ ]:
# ── Load & clean ───────────────────────────────────────────────────────────
dep = load_and_clean(DATA_DIR / 'departees_eastern_drc.csv', EASTERN_PROVINCES, CAUSE_TRANSLATIONS)
ret = load_and_clean(DATA_DIR / 'returnees_eastern_drc.csv', EASTERN_PROVINCES, CAUSE_TRANSLATIONS)

print(f'Filtered departees: {len(dep):,}')
print(f'Filtered returnees: {len(ret):,}')
print(f'Date range: {dep["snapshot_month"].min().strftime("%b %Y")} → {dep["snapshot_month"].max().strftime("%b %Y")}')


In [ ]:
# ── Fig 1: Net monthly IDP flow (displaced − returnees) ───────────────────
dep_monthly = monthly_totals(dep)
ret_monthly = monthly_totals(ret)
net_monthly = dep_monthly.subtract(ret_monthly, fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

colors = ['#E24B4A' if v >= 0 else '#378ADD' for v in net_monthly]
ax.bar(net_monthly.index, net_monthly / 1e3, width=20, color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

ax.set_title('Net Monthly IDP Flow — Eastern DRC\n(positive = net displacement, negative = net return)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Net persons (thousands)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.0f}K'))
ax.grid(axis='y', alpha=0.3)

# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#E24B4A', alpha=0.85, label='Net displacement (more fled than returned)'),
    Patch(color='#378ADD', alpha=0.85, label='Net return (more returned than fled)'),
], fontsize=9, framealpha=0.85)

plt.tight_layout()
save_fig(fig, OUT_DIR / 'idp_net_flow_monthly.png')


In [ ]:
# ── Fig 2: Total displaced vs returnees by province ───────────────────────
dep_prov = dep.groupby('admin1_label')['person'].sum().sort_values()
ret_prov = ret.groupby('admin1_label')['person'].sum().reindex(dep_prov.index).fillna(0)

x     = np.arange(len(dep_prov))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.barh(x + width/2, dep_prov / 1e6, width, color='#E24B4A', alpha=0.85, label='Displaced')
ax.barh(x - width/2, ret_prov / 1e6, width, color='#378ADD', alpha=0.85, label='Returnees')
ax.set_yticks(x)
ax.set_yticklabels(dep_prov.index)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.set_title('Total Displaced vs Returnees by Province (2021–2026)', fontsize=13, fontweight='bold')
ax.set_xlabel('People (millions)')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig, OUT_DIR / 'displaced_returnees_by_province.png')


In [ ]:
# ── Fig 3: Top displacement causes ────────────────────────────────────────
causes = dep['cause_label'].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.barh(causes.index, causes.values / 1e3, color='#E24B4A', alpha=0.85)
ax.set_title('Top Causes of Displacement — Eastern DRC (2021–2026)', fontsize=13, fontweight='bold')
ax.set_xlabel('Displacement events (thousands)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}K'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig, OUT_DIR / 'displacement_causes_top10.png')


In [ ]:
# ── Summary stats PNG ─────────────────────────────────────────────────────
make_summary_png(dep, ret, OUT_DIR / 'displacement_summary_stats.png')
